# Digital Forensics & Malware Analysis Toolkit

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib
import base64
import json
import re
from datetime import datetime, timedelta
import random
import string
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

class DigitalForensicsAnalyzer:
    def __init__(self):
        self.malware_signatures = {
            'trojan': ['backdoor', 'remote_access', 'keylogger', 'data_theft'],
            'ransomware': ['encrypt', 'payment', 'bitcoin', 'decrypt_key'],
            'botnet': ['command_control', 'ddos', 'spam', 'proxy'],
            'spyware': ['monitor', 'screenshot', 'microphone', 'webcam'],
            'adware': ['popup', 'redirect', 'advertisement', 'tracking']
        }

        self.suspicious_processes = [
            'powershell.exe', 'cmd.exe', 'wscript.exe', 'cscript.exe',
            'regsvr32.exe', 'rundll32.exe', 'svchost.exe', 'explorer.exe'
        ]

        self.ioc_patterns = {
            'ip_addresses': r'\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b',
            'domains': r'\b[a-zA-Z0-9][a-zA-Z0-9\-]*[a-zA-Z0-9]*\.[a-zA-Z]{2,}\b',
            'urls': r'https?://[^\s<>"{}|\\^`\[\]]+',
            'email': r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
            'file_paths': r'[A-Za-z]:\\(?:[^\\/:*?"<>|\r\n]+\\)*[^\\/:*?"<>|\r\n]*',
            'registry_keys': r'HKEY_[A-Z_]+\\[^\\]+'
        }

        self.entropy_threshold = 7.5  # High entropy indicates encryption/packing

    def generate_sample_file_data(self, num_files=500):
        """Generate sample file metadata for analysis"""
        file_types = ['.exe', '.dll', '.bat', '.ps1', '.vbs', '.scr', '.pdf', '.doc', '.zip']
        locations = [
            'C:\\Windows\\System32\\',
            'C:\\Windows\\Temp\\',
            'C:\\Users\\Public\\',
            'C:\\ProgramData\\',
            'C:\\Program Files\\',
            '%APPDATA%\\',
            '%TEMP%\\',
            'C:\\Users\\Downloads\\'
        ]

        files = []
        for i in range(num_files):
            # 80% benign, 20% suspicious
            is_malicious = random.random() < 0.2

            if is_malicious:
                # Generate suspicious file characteristics
                filename = ''.join(random.choices(string.ascii_lowercase + string.digits, k=random.randint(8, 16)))
                extension = random.choice(['.exe', '.bat', '.scr', '.vbs'])
                location = random.choice(['C:\\Windows\\Temp\\', 'C:\\Users\\Public\\', '%TEMP%\\'])
                size = random.randint(1024, 5*1024*1024)  # 1KB to 5MB
                entropy = random.uniform(7.0, 8.0)  # High entropy (packed/encrypted)
                digital_signature = False
                creation_time = datetime.now() - timedelta(days=random.randint(0, 30))

                # Add suspicious behaviors
                network_connections = random.randint(5, 50)
                registry_modifications = random.randint(10, 100)
                file_operations = random.randint(20, 200)

            else:
                # Generate benign file characteristics
                filename = random.choice(['notepad', 'calculator', 'explorer', 'chrome', 'firefox', 'office'])
                extension = random.choice(['.exe', '.dll', '.pdf', '.doc'])
                location = random.choice(['C:\\Program Files\\', 'C:\\Windows\\System32\\'])
                size = random.randint(10*1024, 100*1024*1024)  # 10KB to 100MB
                entropy = random.uniform(5.0, 7.0)  # Normal entropy
                digital_signature = random.choice([True, False])
                creation_time = datetime.now() - timedelta(days=random.randint(30, 365))

                network_connections = random.randint(0, 5)
                registry_modifications = random.randint(0, 10)
                file_operations = random.randint(0, 50)

            # Generate file hash
            file_content = filename + str(size) + str(creation_time)
            file_hash = hashlib.sha256(file_content.encode()).hexdigest()

            files.append({
                'filename': filename + extension,
                'full_path': location + filename + extension,
                'size_bytes': size,
                'creation_time': creation_time,
                'modification_time': creation_time + timedelta(minutes=random.randint(0, 60)),
                'sha256_hash': file_hash,
                'entropy': entropy,
                'digital_signature': digital_signature,
                'network_connections': network_connections,
                'registry_modifications': registry_modifications,
                'file_operations': file_operations,
                'is_malicious': is_malicious
            })

        return pd.DataFrame(files)

    def generate_network_traffic(self, num_connections=1000):
        """Generate sample network traffic data"""
        protocols = ['TCP', 'UDP', 'HTTP', 'HTTPS', 'DNS', 'FTP']

        # Benign destinations
        benign_domains = [
            'google.com', 'microsoft.com', 'github.com', 'stackoverflow.com',
            'amazon.com', 'cloudflare.com', 'cdn.jsdelivr.net'
        ]

        # Suspicious destinations (C2 servers, malware domains)
        suspicious_domains = [
            'evil-c2-server.com', 'malware-download.net', 'phishing-site.org',
            'crypto-miner.co', 'botnet-command.ru', 'data-exfil.tk'
        ]

        traffic = []
        for i in range(num_connections):
            is_suspicious = random.random() < 0.15  # 15% suspicious traffic

            if is_suspicious:
                destination = random.choice(suspicious_domains)
                port = random.choice([6667, 8080, 4444, 9999, 1337])  # Common malware ports
                protocol = random.choice(['TCP', 'HTTP'])
                bytes_sent = random.randint(1024, 10*1024*1024)  # Larger data transfers
                bytes_received = random.randint(100, 1024*1024)
                connection_duration = random.randint(60, 3600)  # Longer connections

            else:
                destination = random.choice(benign_domains)
                port = random.choice([80, 443, 53, 21, 25])  # Common legitimate ports
                protocol = random.choice(['HTTP', 'HTTPS', 'DNS'])
                bytes_sent = random.randint(100, 1024*1024)
                bytes_received = random.randint(1024, 10*1024*1024)
                connection_duration = random.randint(1, 300)

            timestamp = datetime.now() - timedelta(minutes=random.randint(0, 1440))

            traffic.append({
                'timestamp': timestamp,
                'source_ip': f"192.168.1.{random.randint(100, 200)}",
                'destination': destination,
                'destination_port': port,
                'protocol': protocol,
                'bytes_sent': bytes_sent,
                'bytes_received': bytes_received,
                'duration_seconds': connection_duration,
                'is_suspicious': is_suspicious
            })

        return pd.DataFrame(traffic)

    def calculate_file_entropy(self, data):
        """Calculate Shannon entropy of data"""
        if not data:
            return 0

        # Count frequency of each byte value
        byte_counts = Counter(data.encode() if isinstance(data, str) else data)
        data_len = len(data)

        # Calculate entropy
        entropy = 0
        for count in byte_counts.values():
            p = count / data_len
            if p > 0:
                entropy -= p * np.log2(p)

        return entropy

    def analyze_file_behavior(self, files_df):
        """Analyze file behavior patterns for malware detection"""
        suspicious_files = []

        for idx, file_info in files_df.iterrows():
            suspicion_score = 0
            reasons = []

            # High entropy (packed/encrypted)
            if file_info['entropy'] > self.entropy_threshold:
                suspicion_score += 3
                reasons.append("High entropy (possibly packed/encrypted)")

            # No digital signature
            if not file_info['digital_signature']:
                suspicion_score += 2
                reasons.append("No digital signature")

            # Suspicious location
            suspicious_locations = ['temp', 'public', 'appdata']
            if any(loc in file_info['full_path'].lower() for loc in suspicious_locations):
                suspicion_score += 2
                reasons.append("Suspicious file location")

            # High network activity
            if file_info['network_connections'] > 10:
                suspicion_score += 3
                reasons.append("Excessive network connections")

            # Registry modifications
            if file_info['registry_modifications'] > 20:
                suspicion_score += 2
                reasons.append("Many registry modifications")

            # File operations
            if file_info['file_operations'] > 100:
                suspicion_score += 2
                reasons.append("Excessive file operations")

            # Recent creation
            if (datetime.now() - file_info['creation_time']).days < 7:
                suspicion_score += 1
                reasons.append("Recently created")

            if suspicion_score >= 5:  # Threshold for flagging as suspicious
                suspicious_files.append({
                    'filename': file_info['filename'],
                    'full_path': file_info['full_path'],
                    'suspicion_score': suspicion_score,
                    'reasons': reasons,
                    'sha256_hash': file_info['sha256_hash'],
                    'entropy': file_info['entropy'],
                    'actual_malicious': file_info['is_malicious']
                })

        return pd.DataFrame(suspicious_files)

    def extract_iocs(self, text_data):
        """Extract Indicators of Compromise from text data"""
        iocs = defaultdict(list)

        for ioc_type, pattern in self.ioc_patterns.items():
            matches = re.findall(pattern, text_data, re.IGNORECASE)
            if matches:
                # Ensure all matches are strings (handle tuple results from complex regex)
                clean_matches = []
                for match in matches:
                    if isinstance(match, tuple):
                        # Take the full match (first element) if it's a tuple
                        clean_matches.append(match[0] if match[0] else str(match))
                    else:
                        clean_matches.append(str(match))
                iocs[ioc_type].extend(clean_matches)

        return dict(iocs)

    def analyze_network_behavior(self, traffic_df):
        """Analyze network traffic for suspicious patterns"""
        analysis = {}

        # Data exfiltration detection (large outbound transfers)
        outbound_threshold = traffic_df['bytes_sent'].quantile(0.95)
        data_exfil_candidates = traffic_df[traffic_df['bytes_sent'] > outbound_threshold]

        # Command & Control detection (regular beaconing)
        c2_candidates = traffic_df.groupby('destination').agg({
            'timestamp': 'count',
            'duration_seconds': 'mean',
            'bytes_sent': 'mean'
        }).rename(columns={'timestamp': 'connection_count'})

        # Filter for potential C2 (many connections, consistent timing)
        c2_candidates = c2_candidates[
            (c2_candidates['connection_count'] > 10) &
            (c2_candidates['duration_seconds'] > 60)
        ]

        # Suspicious ports
        suspicious_ports = [6667, 4444, 1337, 8080, 9999]
        suspicious_port_traffic = traffic_df[traffic_df['destination_port'].isin(suspicious_ports)]

        # Traffic volume analysis
        hourly_traffic = traffic_df.set_index('timestamp').resample('H').agg({
            'bytes_sent': 'sum',
            'bytes_received': 'sum'
        })

        analysis = {
            'total_connections': len(traffic_df),
            'unique_destinations': traffic_df['destination'].nunique(),
            'data_exfiltration_candidates': len(data_exfil_candidates),
            'c2_candidates': len(c2_candidates),
            'suspicious_port_connections': len(suspicious_port_traffic),
            'top_destinations': traffic_df['destination'].value_counts().head(10),
            'protocol_distribution': traffic_df['protocol'].value_counts(),
            'hourly_traffic': hourly_traffic
        }

        return analysis, data_exfil_candidates, c2_candidates

    def generate_incident_timeline(self, files_df, traffic_df):
        """Create an incident timeline combining file and network events"""
        events = []

        # File creation events
        for idx, file_info in files_df.iterrows():
            if file_info['is_malicious']:
                events.append({
                    'timestamp': file_info['creation_time'],
                    'event_type': 'File Creation',
                    'description': f"Malicious file created: {file_info['filename']}",
                    'severity': 'High' if file_info['entropy'] > 7.5 else 'Medium',
                    'source': file_info['full_path']
                })

        # Network events
        suspicious_traffic = traffic_df[traffic_df['is_suspicious']]
        for idx, conn in suspicious_traffic.iterrows():
            events.append({
                'timestamp': conn['timestamp'],
                'event_type': 'Network Communication',
                'description': f"Suspicious connection to {conn['destination']}:{conn['destination_port']}",
                'severity': 'High' if conn['bytes_sent'] > 1024*1024 else 'Medium',
                'source': conn['source_ip']
            })

        timeline_df = pd.DataFrame(events).sort_values('timestamp')
        return timeline_df

    def create_forensics_dashboard(self, files_df, traffic_df, suspicious_files, network_analysis):
        """Create comprehensive forensics visualization dashboard"""
        fig, axes = plt.subplots(3, 3, figsize=(20, 15))
        fig.suptitle('Digital Forensics & Malware Analysis Dashboard', fontsize=16, fontweight='bold')

        # 1. File Type Distribution
        file_extensions = files_df['filename'].str.extract(r'\.([^.]+)$')[0].value_counts()
        axes[0, 0].pie(file_extensions.values, labels=file_extensions.index, autopct='%1.1f%%')
        axes[0, 0].set_title('File Type Distribution')

        # 2. Entropy Distribution
        axes[0, 1].hist([files_df[files_df['is_malicious']]['entropy'],
                         files_df[~files_df['is_malicious']]['entropy']],
                        bins=20, alpha=0.7, label=['Malicious', 'Benign'], color=['red', 'green'])
        axes[0, 1].axvline(x=self.entropy_threshold, color='orange', linestyle='--', label='Threshold')
        axes[0, 1].set_title('File Entropy Distribution')
        axes[0, 1].set_xlabel('Entropy')
        axes[0, 1].set_ylabel('Count')
        axes[0, 1].legend()

        # 3. Suspicious Files by Location
        if not suspicious_files.empty:
            locations = suspicious_files['full_path'].str.extract(r'([A-Za-z]:\\[^\\]+\\)')[0].value_counts()
            axes[0, 2].bar(range(len(locations)), locations.values)
            axes[0, 2].set_title('Suspicious Files by Location')
            axes[0, 2].set_xlabel('Location')
            axes[0, 2].set_ylabel('Count')
            axes[0, 2].set_xticks(range(len(locations)))
            axes[0, 2].set_xticklabels([loc[:20] + '...' if len(loc) > 20 else loc for loc in locations.index], rotation=45)

        # 4. Network Protocol Distribution
        protocol_counts = network_analysis['protocol_distribution']
        axes[1, 0].bar(protocol_counts.index, protocol_counts.values, color='skyblue')
        axes[1, 0].set_title('Network Protocol Distribution')
        axes[1, 0].set_xlabel('Protocol')
        axes[1, 0].set_ylabel('Connections')

        # 5. Top Network Destinations
        top_destinations = network_analysis['top_destinations'].head(8)
        axes[1, 1].barh(range(len(top_destinations)), top_destinations.values)
        axes[1, 1].set_title('Top Network Destinations')
        axes[1, 1].set_xlabel('Number of Connections')
        axes[1, 1].set_yticks(range(len(top_destinations)))
        axes[1, 1].set_yticklabels([dest[:25] + '...' if len(dest) > 25 else dest for dest in top_destinations.index])

        # 6. File Size vs Network Activity
        axes[1, 2].scatter(files_df['size_bytes'], files_df['network_connections'],
                          c=['red' if mal else 'blue' for mal in files_df['is_malicious']],
                          alpha=0.6)
        axes[1, 2].set_title('File Size vs Network Activity')
        axes[1, 2].set_xlabel('File Size (bytes)')
        axes[1, 2].set_ylabel('Network Connections')
        axes[1, 2].set_xscale('log')

        # 7. Malware Detection Accuracy
        if not suspicious_files.empty:
            true_positives = len(suspicious_files[suspicious_files['actual_malicious']])
            false_positives = len(suspicious_files[~suspicious_files['actual_malicious']])
            total_malicious = len(files_df[files_df['is_malicious']])
            false_negatives = total_malicious - true_positives

            detection_data = [true_positives, false_positives, false_negatives]
            labels = ['True Positives', 'False Positives', 'False Negatives']
            colors = ['green', 'orange', 'red']

            axes[2, 0].bar(labels, detection_data, color=colors)
            axes[2, 0].set_title('Malware Detection Results')
            axes[2, 0].set_ylabel('Count')

            # Calculate metrics
            precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
            recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
            axes[2, 0].text(0.5, max(detection_data) * 0.8,
                           f'Precision: {precision:.2f}\nRecall: {recall:.2f}',
                           ha='center', bbox=dict(boxstyle='round', facecolor='wheat'))

        # 8. Network Traffic Timeline
        hourly_traffic = network_analysis['hourly_traffic']
        if not hourly_traffic.empty:
            axes[2, 1].plot(hourly_traffic.index, hourly_traffic['bytes_sent'],
                           label='Outbound', color='red', linewidth=2)
            axes[2, 1].plot(hourly_traffic.index, hourly_traffic['bytes_received'],
                           label='Inbound', color='blue', linewidth=2)
            axes[2, 1].set_title('Network Traffic Over Time')
            axes[2, 1].set_xlabel('Time')
            axes[2, 1].set_ylabel('Bytes')
            axes[2, 1].legend()
            axes[2, 1].tick_params(axis='x', rotation=45)

        # 9. Threat Intelligence Summary
        threat_summary = {
            'Files Analyzed': len(files_df),
            'Malicious Files': len(files_df[files_df['is_malicious']]),
            'Suspicious Files': len(suspicious_files),
            'Network Connections': network_analysis['total_connections'],
            'C2 Candidates': network_analysis['c2_candidates'],
            'Data Exfil Candidates': network_analysis['data_exfiltration_candidates']
        }

        y_pos = np.arange(len(threat_summary))
        axes[2, 2].barh(y_pos, list(threat_summary.values()), color='lightcoral')
        axes[2, 2].set_title('Threat Intelligence Summary')
        axes[2, 2].set_yticks(y_pos)
        axes[2, 2].set_yticklabels(list(threat_summary.keys()))
        axes[2, 2].set_xlabel('Count')

        # Add values on bars
        for i, v in enumerate(threat_summary.values()):
            axes[2, 2].text(v + max(threat_summary.values()) * 0.01, i, str(v),
                           va='center', fontweight='bold')

        plt.tight_layout()
        plt.show()

    def generate_forensics_report(self, files_df, traffic_df, suspicious_files, network_analysis):
        """Generate comprehensive forensics report"""
        total_files = len(files_df)
        malicious_files = len(files_df[files_df['is_malicious']])
        detected_suspicious = len(suspicious_files)

        # Calculate detection metrics
        if not suspicious_files.empty:
            true_positives = len(suspicious_files[suspicious_files['actual_malicious']])
            false_positives = len(suspicious_files[~suspicious_files['actual_malicious']])
            false_negatives = malicious_files - true_positives

            precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
            recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
            f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        else:
            precision = recall = f1_score = 0
            true_positives = false_positives = false_negatives = 0

        report = {
            'executive_summary': {
                'total_files_analyzed': total_files,
                'malicious_files_found': malicious_files,
                'suspicious_files_detected': detected_suspicious,
                'network_connections_analyzed': network_analysis['total_connections'],
                'c2_communications_detected': network_analysis['c2_candidates'],
                'data_exfiltration_attempts': network_analysis['data_exfiltration_candidates']
            },
            'detection_performance': {
                'true_positives': true_positives,
                'false_positives': false_positives,
                'false_negatives': false_negatives,
                'precision': precision,
                'recall': recall,
                'f1_score': f1_score
            },
            'network_analysis': network_analysis,
            'high_risk_files': suspicious_files[suspicious_files['suspicion_score'] >= 8].to_dict('records') if not suspicious_files.empty else [],
            'recommendations': self._generate_recommendations(suspicious_files, network_analysis)
        }

        return report

    def _generate_recommendations(self, suspicious_files, network_analysis):
        """Generate security recommendations based on analysis"""
        recommendations = []

        if not suspicious_files.empty:
            high_entropy_files = len(suspicious_files[suspicious_files['entropy'] > 7.5])
            if high_entropy_files > 0:
                recommendations.append(f"🔍 Investigate {high_entropy_files} high-entropy files - likely packed/encrypted malware")

            unsigned_files = len(suspicious_files[suspicious_files['reasons'].str.contains('No digital signature', na=False)])
            if unsigned_files > 0:
                recommendations.append(f"⚠️ {unsigned_files} suspicious files lack digital signatures - implement code signing policies")

        if network_analysis['c2_candidates'] > 0:
            recommendations.append(f"🚨 {network_analysis['c2_candidates']} potential C2 communications detected - block suspicious domains")

        if network_analysis['data_exfiltration_candidates'] > 0:
            recommendations.append(f"📤 {network_analysis['data_exfiltration_candidates']} potential data exfiltration attempts - review DLP policies")

        if network_analysis['suspicious_port_connections'] > 0:
            recommendations.append(f"🔒 {network_analysis['suspicious_port_connections']} connections to suspicious ports - consider firewall rules")

        recommendations.extend([
            "🛡️ Implement real-time file monitoring for high-entropy files",
            "🔄 Update malware signatures and behavioral detection rules",
            "📊 Establish baseline network traffic patterns for anomaly detection",
            "🎯 Conduct threat hunting exercises based on identified IOCs"
        ])

        return recommendations

def main():
    print("🔍 Digital Forensics & Malware Analysis Toolkit")
    print("=" * 60)

    # Initialize analyzer
    analyzer = DigitalForensicsAnalyzer()

    # Generate sample data
    print("📁 Generating sample file system data...")
    files_df = analyzer.generate_sample_file_data(800)
    print(f"Generated metadata for {len(files_df)} files")

    print("\n🌐 Generating network traffic data...")
    traffic_df = analyzer.generate_network_traffic(1200)
    print(f"Generated {len(traffic_df)} network connections")

    # Analyze file behavior
    print("\n🔬 Analyzing file behaviors for malware indicators...")
    suspicious_files = analyzer.analyze_file_behavior(files_df)
    print(f"Identified {len(suspicious_files)} suspicious files")

    # Analyze network behavior
    print("\n📡 Analyzing network traffic patterns...")
    network_analysis, data_exfil, c2_candidates = analyzer.analyze_network_behavior(traffic_df)

    # Generate incident timeline
    print("\n⏰ Creating incident timeline...")
    timeline = analyzer.generate_incident_timeline(files_df, traffic_df)

    # Generate comprehensive report
    print("\n📋 Generating forensics report...")
    forensics_report = analyzer.generate_forensics_report(files_df, traffic_df, suspicious_files, network_analysis)

    # Display executive summary
    print("\n" + "="*60)
    print("DIGITAL FORENSICS ANALYSIS SUMMARY")
    print("="*60)

    summary = forensics_report['executive_summary']
    print(f"📊 Files Analyzed: {summary['total_files_analyzed']:,}")
    print(f"🦠 Malicious Files Found: {summary['malicious_files_found']}")
    print(f"⚠️  Suspicious Files Detected: {summary['suspicious_files_detected']}")
    print(f"🌐 Network Connections: {summary['network_connections_analyzed']:,}")
    print(f"🎯 C2 Communications: {summary['c2_communications_detected']}")
    print(f"📤 Data Exfiltration Attempts: {summary['data_exfiltration_attempts']}")

    # Display detection performance
    performance = forensics_report['detection_performance']
    print(f"\n🎯 Detection Performance:")
    print(f"   Precision: {performance['precision']:.2%}")
    print(f"   Recall: {performance['recall']:.2%}")
    print(f"   F1-Score: {performance['f1_score']:.2%}")

    # Display high-risk findings
    high_risk = forensics_report['high_risk_files']
    if high_risk:
        print(f"\n🚨 High-Risk Files Detected ({len(high_risk)}):")
        for file_info in high_risk[:5]:  # Show top 5
            print(f"   • {file_info['filename']} (Score: {file_info['suspicion_score']}/10)")
            print(f"     Reasons: {', '.join(file_info['reasons'][:2])}")

    # Display recent incidents
    recent_incidents = timeline.head(10)
    if not recent_incidents.empty:
        print(f"\n🕐 Recent Security Incidents:")
        for idx, incident in recent_incidents.iterrows():
            print(f"   • [{incident['severity']}] {incident['event_type']}: {incident['description'][:60]}...")

    # IOC Extraction Demo
    print(f"\n🔍 IOC Extraction Example:")
    sample_text = """
    The malware connected to evil-c2-server.com on port 4444
    It also accessed registry key HKEY_LOCAL_MACHINE\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Run
    Files were written to C:\\Windows\\Temp\\malware.exe
    Email communications sent to attacker@malicious-domain.net
    """

    iocs = analyzer.extract_iocs(sample_text)
    for ioc_type, indicators in iocs.items():
        if indicators:
            print(f"   {ioc_type.replace('_', ' ').title()}: {', '.join(indicators[:3])}")

    # Create forensics dashboard
    print(f"\n📊 Creating forensics dashboard...")
    analyzer.create_forensics_dashboard(files_df, traffic_df, suspicious_files, network_analysis)

    # Display recommendations
    print(f"\n💡 Security Recommendations:")
    for i, recommendation in enumerate(forensics_report['recommendations'][:5], 1):
        print(f"   {i}. {recommendation}")

    print(f"\n✅ Digital forensics analysis complete!")
    print(f"📈 Check the dashboard above for detailed visualizations and insights.")

if __name__ == "__main__":
    main()